# Train the LeJEPA encoder on OGBench cube-single, and score the identifiability trend

Stage A of `LEJEPA_RUN.md` (§5) plus the frozen metric suite (§9). One seed, ten epochs —
this is a **trend** run, not a converged one. The question it answers is *"are the
identifiability metrics moving in the right direction, and is the theory's bound tracking
the measured recovery?"*, not *"what is the final number?"*.

Prerequisite: `collect_lejepa_ogbcubesingle.ipynb` has pushed both datasets to HF.

Flow: config → apply → install → download → W&B → pre-flight → smoke → train → **statistics** → trend.

## Where the statistics live

This notebook holds the **encoder-side** statistics: `run_metrics.py`, run once per epoch
checkpoint. They belong here because they need a checkpoint, and because running them per
epoch is what turns a single 10-epoch run into a curve — which is the whole deliverable
when there is one seed and no budget for five.

The **dataset-side** statistics (the phase-3 audit of the marginal and the achieved ρ) ran
in the collection notebook, before the upload.

## What one seed buys, and what it does not

`program_constants.seed_budget` is 5 and this run spends 1. That is a deliberate,
recorded deviation, not an oversight: with `n = 1` seed no metric here carries an error
bar, so **read the shape of each curve, not the value at epoch 10**. The scatter rows
record `seed` and `program_constants` verbatim, so a later multi-seed run appends to the
same file and the two are directly comparable.

## 1. Config

In [ ]:
import os

# --- repo ---
REPO_ROOT = '/workspace/stable-worldmodel'          # ← edit if you cloned it elsewhere

# --- storage (network volume) ---
STABLEWM_HOME = '/workspace'                        # datasets/, checkpoints/ land directly here
SPT_CACHE_DIR = '/workspace/cache/stable-pretraining'  # Lightning .ckpt files, not used by the metrics

# --- Hugging Face ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')                                # ← paste if not a pod env var
HF_REPO_ID = 'quastAI/lejepa-ogbench-cube-single-ou'   # ← from the collection notebook

# --- Weights & Biases ---
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')  # ← paste if not a pod env var
WANDB_ENTITY = 'julian-quast-8-technical-university-of-berlin'                 # ← edit me
WANDB_PROJECT = 'lejepa-identifiability'             # launcher/local.yaml would otherwise force 'stable-wm'

# --- the run ---
PROFILE = 'physical_content'   # must match the dataset that was collected
ENCODER = 'paper_cnn'          # the paper's own pixel encoder; `vit_small` is the labelled second arm
EPOCHS = 10                    # one seed, ten epochs — the trend is the deliverable
SEED = 3072
BATCH_SIZE = 256
NUM_WORKERS = 6
LAMBDA = 5.0e-2                # loss = lambda*SIGReg + (1-lambda)*align, in the PAPER's units
OUTPUT_MODEL_NAME = 'lejepa'
ARM = 'A'                      # LeJEPA-trained encoder

# --- metrics ---
# 20000 is the suite's default. The per-epoch sweep embeds 2 views x (OU set + style probe)
# per checkpoint, so this is the knob that decides how long §10 takes.
METRIC_MAX_SAMPLES = 20_000
EPOCHS_TO_SCORE = list(range(1, EPOCHS + 1))

# --- derived, don't edit ---
SUBDIR = f'{OUTPUT_MODEL_NAME}_s{SEED}'
OGBENCH_DIR = os.path.join(STABLEWM_HOME, 'datasets', 'ogbench')
CKPT_DIR = os.path.join(STABLEWM_HOME, 'checkpoints', OUTPUT_MODEL_NAME)
OUTPUT_DIR = os.path.join(STABLEWM_HOME, 'outputs')
SCATTER_PATH = os.path.join(OUTPUT_DIR, f'lejepa_scatter_{SUBDIR}.jsonl')
OU_NAME = f'ogbench/cube_single_ou_{PROFILE}.lance'
STYLE_NAME = f'ogbench/cube_single_ou_style_{PROFILE}.lance'

## 2. Apply config

In [ ]:
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
os.environ['SPT_CACHE_DIR'] = SPT_CACHE_DIR
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

for path in (STABLEWM_HOME, SPT_CACHE_DIR, OGBENCH_DIR, OUTPUT_DIR):
    os.makedirs(path, exist_ok=True)

os.chdir(REPO_ROOT)  # os.chdir (not `%cd`) so it behaves the same after a kernel restart

print('cwd           =', os.getcwd())
print('STABLEWM_HOME =', os.environ['STABLEWM_HOME'])
print('checkpoints ->', CKPT_DIR)
print('scatter     ->', SCATTER_PATH)
!df -h "$STABLEWM_HOME"

## 3. Torch ≥ 2.5

`transformers` needs `torch>=2.5`; some pods ship 2.4.1. **Restart the kernel if it
upgrades**, then re-run cells 1–2.

In [ ]:
import torch
print('torch before:', torch.__version__)

if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 5):
    !pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    print('Upgraded — RESTART THE KERNEL now, then re-run cells 1-2 before continuing.')
else:
    print('torch already >= 2.5, nothing to do.')

## 4. Install dependencies

In [ ]:
%pip install -q -e '.[train,format]' wandb huggingface_hub

## 5. Download the datasets

Both tables plus both sidecar manifests, straight into `$STABLEWM_HOME/datasets/ogbench/`.
The manifests are not optional: `run_metrics.py` **aborts** on a missing one, because it
reads ρ, the violation key, `config_hash`, `latents.n` and the isotropy margin from there
and guessing a default is how two runs become indistinguishable in the scatter.

Safe to re-run.

In [ ]:
!hf download "$HF_REPO_ID" --repo-type dataset --local-dir "$OGBENCH_DIR"

In [ ]:
import json
from pathlib import Path

import lance

for name in (OU_NAME, STYLE_NAME):
    stem = Path(name).stem
    table = Path(OGBENCH_DIR) / f'{stem}.lance'
    sidecar = Path(OGBENCH_DIR) / f'{stem}_manifest.json'
    assert table.exists(), f'missing dataset {table}'
    assert sidecar.exists(), f'missing manifest {sidecar} — run_metrics.py will abort'
    manifest = json.loads(sidecar.read_text())
    rows = lance.dataset(str(table)).count_rows()
    print(f'{stem:44s} {rows // 2:>8,} pairs  n={manifest["latents"]["n"]:<3d} '
          f'rho={manifest["ou"]["rho_mean"]}  hash={manifest["config_hash"]}')

N_LATENT = json.loads((Path(OGBENCH_DIR) / f'{Path(OU_NAME).stem}_manifest.json').read_text())['latents']['n']
print(f'\nhead width m will be taken from the data: m = n = {N_LATENT}')

## 6. W&B login

In [ ]:
import wandb

wandb.login(key=WANDB_API_KEY)

## 7. Pre-flight

Two gates, both cheap, both before any GPU time is spent.

In [ ]:
assert torch.cuda.is_available(), 'No GPU visible — lejepa.yaml pins accelerator: gpu'
print(torch.cuda.get_device_name(0))
print('devices pinned to 1 on purpose: SIGReg is a batch statistic and is never gathered '
      'across ranks, so two devices would silently halve the effective batch.')

**The V4 calibration.** Pure numpy, no checkpoint, no dataset, no GPU. It scores a
*synthetic* analytically-optimal encoder to prove the frozen metric suite can resolve the
one published quantitative prediction — recovery fails once `min ρ_α ≤ (max ρ_α)²`. If it
fails, the suite cannot measure the effect it was built to measure and every row §10
writes would be uninterpretable. Nothing in the code enforces this dependency; it is a
gate you honour.

In [ ]:
import subprocess
import time


HYDRA_QUIET = ['hydra.run.dir=.', 'hydra.output_subdir=null']


def sh(cmd, check=True):
    """Run a command, stream its output live, raise on a non-zero exit."""
    print('$', ' '.join(cmd), flush=True)
    started = time.time()
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end='')
    elapsed = time.time() - started
    print(f'\n[{elapsed / 60:.1f} min]')
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed with exit code {proc.returncode}')
    return proc.returncode


sh(['python', 'scripts/identifiability/run_v4_calibration.py',
    f'n={N_LATENT}', 'rho=0.9', f'seed={SEED}',
    f'output={os.path.join(OUTPUT_DIR, "v4_calibration.json")}',
    *HYDRA_QUIET])

## 8. Smoke train

Twenty batches, one epoch, W&B off, into a throwaway checkpoint name. It surfaces a shape
problem, a bad `latent/z` width or an OOM in a couple of minutes.

`num_sanity_val_steps=1` is hardcoded in the script, so one validation batch runs before
any training step.

Two things to read out of it before spending the real run: the progress bar should carry
`fit/recovery/*` and `fit/spectrum/*` keys — if it does not, this checkout predates the
diagnostics and §9's kill rules will not be available — and `fit/bound/trace_cov` should
already be moving off ~1.5. There is **no `lr` here**: `LearningRateMonitor` needs a
logger and W&B is off for the smoke run, so the schedule can only be checked in §9.

In [ ]:
sh(['python', 'scripts/train/lejepa.py',
    f'profile={PROFILE}',
    f'encoder={ENCODER}',
    f'output_model_name={OUTPUT_MODEL_NAME}_smoke',
    'subdir=smoke',
    'trainer.max_epochs=1',
    # `+` because cfg.trainer is a struct: these two keys are not in lejepa.yaml
    '+trainer.limit_train_batches=20',
    '+trainer.limit_val_batches=5',
    f'loader.batch_size={BATCH_SIZE}',
    'loader.num_workers=2',
    f'program_constants.lambda={LAMBDA}',
    'wandb.enabled=false',
    *HYDRA_QUIET])

## 9. Train — 10 epochs, one seed

Launched detached, so it survives closing the browser or the kernel; only a pod stop kills it.

### What is logged, and what each one catches

Everything below is computed under `no_grad` and **never optimised**. That is not a
detail: `epsilon` is only an honest measurement of the theory's bound because it is
independent of what is being minimised, and the same argument covers the rest —
`recovery/*` in particular scores against `latent/z`, which is a *label*. It has always
been in every batch (the data config loads it); the forward simply never read it. Putting
it in the objective would turn an identifiability claim into supervised regression.

The "step 0" column is measured, not guessed: a randomly initialised `paper_cnn` at
`n=10`, `B=256`, 224px, λ=5e-2 — i.e. what epoch 0 of this exact run looks like.

| key | step 0 | healthy | what a bad value means |
|---|---|---|---|
| `fit/loss` | 7.02 | falls | `λ·sigreg + (1−λ)·align`. |
| `fit/sigreg_loss` | 139 | **falls hard and fast** | The Epps–Pulley isotropy statistic — the only thing preventing collapse; there is no EMA target. It should drop by ~two orders inside a few hundred steps. |
| `fit/align_loss` | 0.0697 | falls | `(h.mean(0) − h)²`. The views share content up to one OU step and differ entirely in style, so the only way down is to stop representing style. |
| `fit/balance/sigreg_share` | 0.991 | falls well below 1 | The share of the objective SIGReg accounts for. Pinned near 1 *while* `align_loss` is flat is λ too high — drop to 1e-2. Early dominance is expected and fine. |
| `fit/whitening_metric` | 0.0763 | drifts down | `‖Cov−I‖²_F/n²`. **Note how small it already is** — see the spectrum row. |
| `fit/bound/epsilon` | 2.75 | falls | The same quantity in the bound's units, `‖Cov−I‖_F`. |
| `fit/bound/trace_cov` | 1.50 | rises toward `n`=10 | Heading to 0 is total collapse. This is the fastest collapse detector there is. |
| `fit/bound/delta` | 2.49 | falls | The excess pair distance the OU step does not account for. Plateauing above 0 while `epsilon` is small is style reaching the output. |
| `fit/bound/D`, `predicted_error`, `bound_headroom` | 13.8 / 288 / −278 | headroom rises toward 0+ | The theory's own prediction. Headroom is `n − predicted`: negative means the bound exceeds `E‖z‖²=n`, which `h=0` already achieves, so it is saying nothing yet. |
| `fit/spectrum/cov_eig_min` | 0.0261 | rises toward 1 | Smallest eigenvalue of `Cov(h)`. Zero is a direction that carries nothing. |
| `fit/spectrum/effective_rank` | **4.69** | rises toward `n`=10 | `(Σλ)²/Σλ²` — how many directions actually carry variance. |
| `fit/recovery/r2_z_to_h` | 0.019 | rises | Linear R², `z→h`. |
| `fit/recovery/r2_h_to_z` | 0.027 | rises | Linear R², `h→z`. |
| `fit/recovery/procrustes_mse_per_dim` | 1.03 | falls | **The criterion metric**, the same number the scatter calls `measured_recovery_error`. |
| `fit/recovery/orth_err_normalized` | 0.997 | falls | `‖AᵀA−I‖_F/√n` of the best linear map. |
| `fit/recovery/cond` | 70.0 | falls | Condition number of that map — the one that matters for planning. |
| `lr-AdamW` | 0 → peak | rises through warmup, then cosine | From `LearningRateMonitor`. |

> **Why the spectrum row is not redundant with `whitening_metric`.** At step 0 the
> whitening metric reads 0.076, which looks like nothing — while the embedding is living
> in **4.7 of its 10 directions**. `epsilon` is a Frobenius aggregate and one dead
> direction contributes 1 to `epsilon²` out of `n`; it disappears into ordinary
> early-training values. `cov_eig_min` and `effective_rank` do not average it away, and
> partial collapse is the failure mode most likely to survive to epoch 10 while every
> loss curve looks fine.

### Kill rules — the point of all this

The realistic saving is not early-stopping a *converging* run; it is killing a **broken**
one in epoch 1 instead of at epoch 10. Check the log after ~200 steps and again at the
end of epoch 1:

1. **`lr-AdamW` is exactly 0 for the whole first epoch** → the schedule regressed to
   `interval: 'epoch'`. This has happened in this repo before (§12 defect 6) and cost a
   run that looked like it trained. Kill immediately.
2. **`sigreg_loss` flat near 139, or `trace_cov` → 0** → nothing is being regularised, or
   the representation is collapsing outright. Kill.
3. **`effective_rank` below its step-0 4.69 and still falling at the end of epoch 1** →
   SIGReg is losing. Kill; re-run at a higher λ.
4. **`r2_z_to_h` still under ~0.1 at the end of epoch 2, with everything else moving** →
   the losses are optimising and the latents are not being recovered. That is the one
   failure no loss curve shows, and it is why these are logged.
5. **`balance/sigreg_share` pinned at ~1 while `align_loss` is flat** → λ too high. Kill
   and re-run at 1e-2 rather than spending ten epochs on it.

`r2_h_to_z` climbing while `r2_z_to_h` stays flat is **not** a kill — it is the V4
second-Hermite signature, and at severity 0 it is a finding worth having.

> These diagnostics live in `wm/lejepa/losses.py` (`recovery_diagnostics`,
> `spectrum_diagnostics`) and are wired in `scripts/train/lejepa.py`. Make sure the pod's
> checkout has them, or the run will log only the original nine keys.

In [ ]:
log_path = os.path.join(STABLEWM_HOME, 'logs', f'{SUBDIR}.log')
os.makedirs(os.path.dirname(log_path), exist_ok=True)

cmd = [
    'python', 'scripts/train/lejepa.py',
    f'profile={PROFILE}',
    f'encoder={ENCODER}',
    f'output_model_name={OUTPUT_MODEL_NAME}',
    f'subdir={SUBDIR}',
    f'seed={SEED}',
    f'trainer.max_epochs={EPOCHS}',
    f'loader.batch_size={BATCH_SIZE}',
    f'loader.num_workers={NUM_WORKERS}',
    f'program_constants.lambda={LAMBDA}',
    'wandb.enabled=true',
    f'wandb.config.entity={WANDB_ENTITY}',
    f'wandb.config.project={WANDB_PROJECT}',
    *HYDRA_QUIET,
]
print('$', ' '.join(cmd))

with open(log_path, 'w') as handle:
    train_proc = subprocess.Popen(
        cmd, stdout=handle, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        start_new_session=True,  # detaches from this kernel's process group
    )

print('\nStarted PID', train_proc.pid)
print('Log:', log_path)

Poll until it exits. Interrupting this cell does **not** stop the run — it is detached. Re-running the cell resumes watching, but only while this kernel lives: after a restart `train_proc` is gone and the log tail below is the way to follow it.

In [ ]:
while train_proc.poll() is None:
    saved = sorted(Path(CKPT_DIR).glob('weights_epoch_*.pt')) if Path(CKPT_DIR).exists() else []
    tail = subprocess.run(['tail', '-n', '1', log_path], capture_output=True, text=True).stdout.strip()
    print(f'[{time.strftime("%H:%M:%S")}] {len(saved)}/{EPOCHS} epochs saved | {tail[-140:]}')
    time.sleep(120)

print('\nexit code', train_proc.returncode)

In [ ]:
!tail -n 40 "$log_path"

In [ ]:
checkpoints = sorted(
    Path(CKPT_DIR).glob('weights_epoch_*.pt'),
    key=lambda p: int(p.stem.rsplit('_', 1)[1]),
)
print(f'{len(checkpoints)} checkpoints in {CKPT_DIR}')
for path in checkpoints:
    print(' ', path.name)

encoder_hash = (Path(STABLEWM_HOME) / 'checkpoints' / SUBDIR / 'encoder_hash.txt').read_text().strip()
print(f'\nencoder hash: {encoder_hash}   ← record this with the run; `seed` does not '
      'make it reproducible, so if you lose the .pt you cannot regenerate the pair.')

# Trainer state (optimizer + scheduler + loops), one per epoch. Its directory is
# chosen by spt.Manager, not by the config -- see §12 -- so the run records it.
pointer = Path(STABLEWM_HOME) / 'checkpoints' / SUBDIR / 'lightning_ckpt_dir.txt'
if pointer.exists():
    lightning_dir = Path(pointer.read_text().strip())
    ckpts = sorted(lightning_dir.glob('*.ckpt'))
    total = sum(c.stat().st_size for c in ckpts) / 1e9
    print(f'\ntrainer state -> {lightning_dir}')
    print(f'  {len(ckpts)} files, {total:.2f} GB: {[c.name for c in ckpts]}')
else:
    print('\nno lightning_ckpt_dir.txt — this checkout predates '
          'RecordCkptDirCallback, so the trainer state is under '
          '$SPT_CACHE_DIR/runs/<date>/<time>/<run_id>/checkpoints/')

## 10. Statistics — the frozen metric suite, once per epoch

`run_metrics.py` scores the **encoder alone**. `LeJEPA` has no dynamics by design: the
script only calls `model.encode(...)['emb']`, and no predictor checkpoint is an input here.

Three things about how this is invoked:

- **`rollout_dataset=null`.** `collect_cube_single_predictor.py` writes no manifest and no
  `latent/z` column, so the rollout half cannot be scored as shipped (`LEJEPA_RUN.md` §12
  defect 4) — and worse, `append_rows` is called once *after* both distributions, so a
  rollout-side failure would discard the already-computed OU row too. Each run therefore
  logs one row, `distribution='ou'`, and says the OU/rollout gap is not measured. That is
  accurate: it isn't.
- **`program_constants.lambda` is passed explicitly.** `metrics.yaml` still defaults to
  `3.0e-3` while `lejepa.yaml` trains at `5.0e-2`, and this value is copied *verbatim*
  into every scatter row. Left alone, the rows would record a λ the encoder was never
  trained at.
- **the style probe is armed** by `metrics.yaml`'s default and resolves from `PROFILE`. It
  is what lets `delta` split into `delta_content` and `delta_style`; without it the bound
  charges style leakage to nonlinearity.

The scatter is append-only. Re-running this cell adds rows rather than replacing them.

In [ ]:
started = time.time()
for epoch in EPOCHS_TO_SCORE:
    checkpoint = Path(CKPT_DIR) / f'weights_epoch_{epoch}.pt'
    if not checkpoint.exists():
        print(f'skipping epoch {epoch}: no checkpoint')
        continue
    print(f'\n===== epoch {epoch} =====')
    sh(['python', 'scripts/identifiability/run_metrics.py',
        f'checkpoint={OUTPUT_MODEL_NAME}/weights_epoch_{epoch}.pt',
        f'arm={ARM}',
        f'profile={PROFILE}',
        'rollout_dataset=null',
        f'max_samples={METRIC_MAX_SAMPLES}',
        f'seed={SEED}',
        'device=cuda',
        f'program_constants.lambda={LAMBDA}',
        f'scatter_path={SCATTER_PATH}',
        *HYDRA_QUIET])

print(f'\nscored {len(EPOCHS_TO_SCORE)} checkpoints in {(time.time() - started) / 60:.1f} min')
print('scatter ->', SCATTER_PATH)

## 11. The trend

The table first — every column of the frozen suite, so nothing is hidden behind the choice
of what to plot.

In [ ]:
import numpy as np
import pandas as pd

rows = [json.loads(line) for line in Path(SCATTER_PATH).read_text().splitlines() if line.strip()]
scatter = pd.DataFrame(rows)
scatter['epoch'] = scatter['checkpoint'].str.extract(r'weights_epoch_(\d+)\.pt').astype(int)

df = (
    scatter[(scatter.distribution == 'ou') & (scatter.seed == SEED) & (scatter.arm == ARM)]
    .drop_duplicates(subset='epoch', keep='last')     # append-only: a re-run adds rows
    .sort_values('epoch')
    .reset_index(drop=True)
)

print(f'{len(df)} epochs   n={df.n.iloc[0]}   profile={df.profile.iloc[0]}   '
      f'config_hash={df.config_hash.iloc[0]}   suite={df.metric_suite_version.iloc[0]}')
print(f'style probe present: {bool(df.has_style_probe.iloc[0])}   '
      f'bound vacuous: {bool(df.bound_vacuous.iloc[-1])}')

pd.set_option('display.width', 220, 'display.max_columns', 60)
df.set_index('epoch')[[
    'measured_recovery_error', 'predicted_error', 'orth_err_normalized', 'cond',
    'r2_z_to_h', 'r2_h_to_z', 'probe_linear_r2', 'probe_divergence',
    'epsilon', 'delta', 'delta_content', 'delta_style',
    'style_sensitivity', 'sigreg_z', 'hermite2_excess', 'mcc_unaligned',
    'spectral_gap', 'recovery_in_gap_units', 'bound_headroom',
]].round(4)

Small multiples, one metric per panel — the honest form when the panels share an x axis
(epoch) and nothing else. Never two y-scales on one panel: the two-series panels below pair
quantities that are already in the same units.

In [ ]:
import matplotlib.pyplot as plt

INK, INK_2, MUTED = '#0b0b0b', '#52514e', '#8a8880'
SURFACE, GRIDLINE = '#fcfcfb', '#e6e5e1'
S1, S2 = '#2a78d6', '#eb6834'   # validated categorical slots 1 and 2

PANELS = [
    ('Recovery error', [('measured_recovery_error', None)],
     'Procrustes MSE per dim — the criterion metric. Down.'),
    ('Measured vs predicted', [('measured_recovery_error', 'measured'), ('predicted_error', 'D + (eps+D)^2')],
     "the theory's bound against what was measured. Same units."),
    ('Orthogonality error', [('orth_err_normalized', None)],
     '||A^T A - I||_F / sqrt(n) of the best linear map. Down.'),
    ('Condition number', [('cond', None)],
     'of the best linear map — the one that matters for planning. Down.'),
    ('Linear R^2, both directions', [('r2_z_to_h', 'z -> h'), ('r2_h_to_z', 'h -> z')],
     'h->z high while z->h collapses is the V4 second-Hermite signature.'),
    ('Probe R^2 and its divergence', [('probe_linear_r2', 'linear probe'), ('probe_divergence', 'probe - recovery')],
     'the decoy metric. Divergence climbing = probe-ability without recovery.'),
    ('epsilon = ||Cov(h) - I||_F', [('epsilon', None)],
     "the bound's first term. Logged during training, never optimised."),
    ('delta, split', [('delta_content', 'nonlinearity'), ('delta_style', 'style leakage')],
     'the style probe is what makes this split possible at all.'),
    ('Style sensitivity', [('style_sensitivity', None)],
     'what the alignment loss is supposed to discard. Down.'),
    ('SIGReg z-score', [('sigreg_z', None)],
     'against a matched i.i.d.-Gaussian null at this sample size. Toward 0.'),
    ('Hermite-2 excess', [('hermite2_excess', None)],
     'positive = the V4 degree-2 substitution is actually happening.'),
    ('Recovery in spectral-gap units', [('recovery_in_gap_units', None)],
     'for comparability across environments. Down.'),
]

available = [p for p in PANELS if all(k in df and df[k].notna().any() for k, _ in p[1])]
skipped = [p[0] for p in PANELS if p not in available]

epochs = df['epoch'].to_numpy()
ncols = 3
nrows = -(-len(available) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.5 * nrows), dpi=130)
fig.patch.set_facecolor(SURFACE)

for ax, (title, series, note) in zip(axes.ravel(), available):
    ax.set_facecolor(SURFACE)
    for (key, label), color in zip(series, (S1, S2)):
        y = df[key].to_numpy(dtype=float)
        ax.plot(epochs, y, color=color, lw=1.8, marker='o', ms=5.5,
                mfc=color, mec=SURFACE, mew=1.2, label=label, zorder=3)
        ax.annotate(f'{y[-1]:.3g}', (epochs[-1], y[-1]), textcoords='offset points',
                    xytext=(7, 0), va='center', fontsize=8.5, color=INK_2)

    ax.set_title(title, fontsize=11, color=INK, loc='left', pad=24)
    ax.text(0.0, 1.035, note, transform=ax.transAxes, fontsize=8.5,
            color=MUTED, ha='left', va='bottom')
    ax.grid(axis='y', color=GRIDLINE, lw=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(GRIDLINE)
    ax.tick_params(colors=MUTED, labelsize=9, length=3)
    ax.set_xticks(epochs)
    span = max(epochs[-1] - epochs[0], 1)
    ax.set_xlim(epochs[0] - 0.3, epochs[-1] + 0.20 * span)
    if len(series) > 1:
        ax.legend(frameon=False, fontsize=8.5, labelcolor=INK_2, loc='best')

for ax in axes.ravel()[len(available):]:
    ax.set_visible(False)
for ax in axes.ravel()[max(0, len(available) - ncols):len(available)]:
    ax.set_xlabel('epoch', fontsize=9, color=INK_2)

fig.suptitle(
    f'LeJEPA identifiability over training — arm {ARM}, {PROFILE} (n={df.n.iloc[0]}), '
    f'{ENCODER}, seed {SEED}, one run',
    fontsize=12.5, color=INK, x=0.012, ha='left', y=0.997,
)
fig.tight_layout(rect=(0, 0, 1, 0.985))

plot_path = os.path.join(OUTPUT_DIR, f'lejepa_trend_{SUBDIR}.png')
fig.savefig(plot_path, dpi=160, facecolor=SURFACE, bbox_inches='tight')
plt.show()

if skipped:
    print('panels skipped (metric absent or all-NaN):', ', '.join(skipped))
print('saved ->', plot_path)

## 12. How to read it, and what is on disk

Most of this was already visible live during training (§9) — `run_metrics.py` differs in
that it scores the full eval set rather than one batch, adds the style probe, and writes
the append-only scatter. If §9's kill rules were watched, nothing here should be a
surprise; if something is, that gap is itself worth recording.

**The shape to look for.** `measured_recovery_error`, `orth_err_normalized` and `epsilon`
falling together, `r2_z_to_h` and `r2_h_to_z` rising together, `style_sensitivity` falling.
`predicted_error` should track `measured_recovery_error` — that is the bound doing its job,
and it is the single most informative panel here.

**The shapes that mean something is wrong.**

- `r2_h_to_z` high while `r2_z_to_h` collapses, with `hermite2_excess` positive: the
  encoder is representing `He₂(z) = z²−1` rather than `z`. That is the V4 signature and it
  is a *finding*, not a bug — at severity 0 it should not appear.
- `probe_divergence` climbing while `orth_err_normalized` is flat: the latents are
  linearly readable but not orthogonally recovered. This is exactly why the probe is
  labelled the decoy metric.
- `fit/align_loss` flat with `sigreg_loss` still falling: λ is too high for this run.
- `bound_vacuous` true: the bound is above `n`, and predicting `h = 0` already achieves
  `n`, so the prediction says nothing at this point in training.

**Two known ceilings, not encoder failures.** The camera is a pick-and-place frame and
cannot be reframed, so `cube.pos_xy[0]` keeps ~27 px of travel against `[1]`'s 96 px, and
`cube.pos_z` less than both. Expect those dimensions to trail, and score them with a stated
ceiling rather than letting them drag the aggregate. Thm 1 quantifies over all measurable
`h`, so framing affects reachability and δ — never the optimum.

**Ten epochs is `V8` at a severity you did not choose.** `trainer.max_epochs` *is* the V8
optimisation-gap knob (`LEJEPA_RUN.md` §10); severity 0 is the full 100-epoch schedule.
Every number here is therefore a truncated-optimisation number, which is precisely why the
trend matters more than the endpoint. `epsilon` being logged-but-never-optimised is what
keeps that question answerable at all.

### Two kinds of checkpoint, and only one of them has a settable path

| | `weights_epoch_N.pt` | `epochNNN.ckpt` |
|---|---|---|
| written by | `SaveCkptCallback` | `ModelCheckpoint` (`checkpoint.keep_every_epoch`) |
| contains | weights only | weights **+** optimizer, scheduler, loops, RNG |
| epoch index | from **1** | from **0** — `epoch004.ckpt` produced `weights_epoch_5.pt` |
| used for | `load_pretrained`, and §10's scoring | resuming a killed run |
| path | `$STABLEWM_HOME/checkpoints/$OUTPUT_MODEL_NAME/` — yours | **not settable**, see below |
| size | ~11 MB (`paper_cnn`) | ~34 MB (`paper_cnn`), ~270 MB (`vit_small`) |

**Why the second path is not settable.** `spt.Manager` always runs in cache_dir mode — a
cache dir is mandatory, passing `None` raises — and `_configure_cache_dir_checkpointing`
rewrites the `dirpath` of *every* `ModelCheckpoint` to
`$SPT_CACHE_DIR/runs/<date>/<time>/<run_id>/checkpoints/`. You will see the redirect in
the training log. What you *can* set is the cache root, `SPT_CACHE_DIR` (cell 1), which is
why it points at the network volume. The `<date>/<time>/<run_id>` tail is not knowable
before the run starts, so `RecordCkptDirCallback` resolves it at **train start** and
writes `lightning_ckpt_dir.txt` plus a `lightning` symlink next to `config.yaml` — at
train start rather than after `fit` returns, because the run whose checkpoint directory
you need to find is precisely the run whose `fit` never returns.

Manager also adds its own rolling `last.ckpt` every epoch regardless of the setting, so
resume works even at `checkpoint.keep_every_epoch=false`; the setting only decides whether
you can go back to a *specific* epoch.

> **Resuming is not wired.** `lejepa.py` passes `ckpt_path=None`, and two traps sit behind
> changing that: `spt.Manager`'s `weights_only` defaults to **True** (optimizer and
> scheduler silently discarded — transfer-init, not resume), and Manager auto-resumes from
> its own `last.ckpt` only when `SLURM_RESTART_COUNT >= 1`, which never happens on RunPod.
> To actually resume, pass the absolute `last.ckpt` path *and* `weights_only=False`.

```
$STABLEWM_HOME/
├── checkpoints/
│   ├── lejepa/weights_epoch_{1..10}.pt   ← one per epoch, never pruned
│   ├── lejepa/config.json                ← cfg.model, what load_pretrained reads
│   └── lejepa_s3072/
│       ├── config.yaml, encoder_hash.txt
│       ├── lightning_ckpt_dir.txt        ← where the trainer state really went
│       └── lightning -> $SPT_CACHE_DIR/runs/<date>/<time>/<run_id>/checkpoints/
│              ├── epoch{000..009}.ckpt   ← full trainer state, one per epoch
│              └── last.ckpt              ← Manager's rolling requeue copy
├── outputs/
│   ├── lejepa_scatter_lejepa_s3072.jsonl ← one row per scored epoch, append-only
│   ├── lejepa_trend_lejepa_s3072.png
│   └── v4_calibration.json
└── logs/lejepa_s3072.log
```

**Next runs that append to the same scatter.** Each is one more `arm=` on the
`run_metrics.py` call, and nothing above changes:

```bash
# the shared-rendering-channel arm (n = 13; cube.color joins z)
python scripts/data/collect_cube_single_ou.py latents.profile=task_content
python scripts/train/lejepa.py profile=task_content
python scripts/identifiability/run_metrics.py arm=A profile=task_content     checkpoint=lejepa/weights_epoch_10.pt rollout_dataset=null

# arms C1/C2 — trajectory-derived pairs instead of designed OU pairs.
# Arm C runs on `arm_c_content` (n = 9): it rebuilds z from compute_ob_info(),
# so every content latent needs a privileged/proprio readback.
python scripts/data/collect_cube_single_arm_c.py arm=c1
```

**Arm R is not one of them, and that is worth being precise about.** `run_metrics.py`
takes an encoder *checkpoint*, and nothing in the repo writes a random-init one — the
literal `encoder=random` is a `lejepa_predictor.py` option, so arm R exists at stage D as
a one-step latent-MSE comparison, not as an identifiability row here. Reading these
numbers against a floor therefore means either producing a random-init checkpoint by hand
(`save_pretrained` on a freshly instantiated `LeJEPA`, same `encoder=` group so the
architectures match) or reading epoch 1 as the closest thing this run has to one.